Step 1: Define File Path and Load Data
This step defines the file path for the 2020 property assessment dataset and checks if the file exists. If found, it loads the dataset, removes any leading/trailing whitespace from column names, and handles missing values for consistency.

In [1]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [9]:
# Define the file path
file_path = r'C:\Users\sul19\Desktop\701 Project\Property Assessment Datasets\Property Assessment 2020 V1.xlsx'

# Check if the file exists before attempting to load it
if os.path.exists(file_path):
    print("File found! Proceeding to load the data.")
    
    # Attempt to load the dataset using 'openpyxl' engine
    df = pd.read_excel(file_path, engine='openpyxl', na_values=['', ' '])

    # Strip any leading/trailing whitespace from the column names (just in case)
    df.columns = df.columns.str.strip()
    
else:
    print(f"File not found at {file_path}. Please check the file path.")

File found! Proceeding to load the data.


Step 2: Standardize Values in OVERALL_COND Column
This step removes non-breaking spaces and other invisible characters from the OVERALL_COND column, then replaces any blank or missing values with "none" to ensure consistency. It also standardizes specific values for improved data quality in the next steps.

In [10]:
# Replace non-breaking spaces and other invisible characters in the overall condition column
df['OVERALL_COND'] = df['OVERALL_COND'].astype(str).str.replace('\u00A0', '').str.strip()

# Replace any blank or missing values with 'none'
df['OVERALL_COND'] = df['OVERALL_COND'].replace(r'^\s*$', 'none', regex=True)

# Replace specific values in OVERALL_COND
df['OVERALL_COND'] = df['OVERALL_COND'].replace({
    'AVG - Default - Average': 'A - Average',
    'EX - Excellent': 'E - Excellent'
}, regex=False)

Step 3: Group Data by ZIP_CODE and Summarize OVERALL_COND Counts
In this step, the data is grouped by ZIP code, counting occurrences of each condition. This summary provides an overview of housing conditions across different areas based on condition counts.

In [11]:
# Group data by 'ZIP_CODE' and count the occurrences of each condition in the column
overall_cond_summary = df.groupby('ZIP_CODE')['OVERALL_COND'].value_counts().unstack().fillna(0)

# Display the result of the analysis
print("Housing condition summary by ZIP code:")
#print(condition_summary)
print(overall_cond_summary)

Housing condition summary by ZIP code:
OVERALL_COND  A - Average  E - Excellent  F - Fair  G - Good  P - Poor  \
ZIP_CODE                                                                 
2018.0                0.0            0.0       0.0       0.0       0.0   
2026.0                6.0            0.0       0.0       1.0       0.0   
2108.0               34.0          103.0       3.0     123.0       0.0   
2109.0               19.0            1.0       1.0       4.0       0.0   
2110.0                0.0            0.0       0.0       0.0       0.0   
2111.0               12.0            0.0       8.0       2.0       0.0   
2112.0                0.0            0.0       0.0       0.0       0.0   
2113.0               82.0            5.0      10.0      30.0       0.0   
2114.0               86.0           32.0       7.0     128.0       3.0   
2115.0               62.0            9.0       2.0      68.0       0.0   
2116.0              157.0          116.0      15.0     184.0       4.0   

Step 4: Map Condition Labels to Numeric Values for Analysis
To facilitate quantitative analysis, this step maps condition labels to numeric values. We then calculate the mean condition score for each ZIP code, offering insights into the average housing condition in each area.

Step 5: Convert Mean Condition Scores to Descriptive Labels
This step converts mean condition scores back to descriptive labels for better readability. Each ZIP code receives a condition label that represents the general state of housing based on its average condition score.

In [12]:
# Function to assign numerical values to conditions 
def condition_to_numeric(cond):
    mapping = {
        'E - Excellent': 5,
        'VG - Very Good': 4,
        'G - Good': 3.5,
        'A - Average': 3,
        'F - Fair': 2,
        'P - Poor': 1.5,
        'VP - Very Poor': 1,
        'US - Unsound': 0,
        
        
        'none': np.nan  # Treat 'none' as NaN for numerical purposes
    }
    return mapping.get(cond, np.nan)

# Apply the mapping to calculate average conditions
df['OVERALL_COND_NUM'] = df['OVERALL_COND'].apply(condition_to_numeric)

# Group by ZIP_CODE and calculate the mean, count, and standard deviation
condition_analysis = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Display the summary
print("Housing condition analysis by ZIP code:")
print(condition_analysis)

def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    
    else:
        return 'Unsound'

# Apply the function to map the means back to descriptive condition labels
condition_analysis['overall_cond_label'] = condition_analysis['overall_cond_mean'].apply(mean_to_condition)

# Print the result for each ZIP code
print("Overall Condition Analysis by ZIP Code:")
print(condition_analysis[['ZIP_CODE', 'overall_cond_mean', 'overall_cond_label']])

Housing condition analysis by ZIP code:
    ZIP_CODE  overall_cond_mean
0     2018.0                NaN
1     2026.0           3.071429
2     2108.0           4.005703
3     2109.0           3.120000
4     2110.0                NaN
5     2111.0           2.681818
6     2112.0                NaN
7     2113.0           3.118110
8     2114.0           3.455078
9     2115.0           3.354610
10    2116.0           3.636555
11    2118.0           3.438287
12    2119.0           3.073384
13    2120.0           3.116228
14    2121.0           3.070842
15    2122.0           3.053440
16    2124.0           3.071562
17    2125.0           3.073534
18    2126.0           3.037037
19    2127.0           3.115514
20    2128.0           3.066502
21    2129.0           3.202652
22    2130.0           3.146547
23    2131.0           3.058716
24    2132.0           3.086375
25    2133.0                NaN
26    2134.0           3.031829
27    2135.0           3.044645
28    2136.0           3.044248


Step 6: Standardize YR_BUILT and YR_REMODEL Columns
This step ensures YR_BUILT and YR_REMODEL columns are numeric and removes any values beyond 2024 to avoid future-dated entries. The mean construction and remodel years by ZIP code are then calculated for further analysis.

Step 7: Classify Buildings by Age
In this step, buildings are classified based on their construction or remodel year as 'Old,' 'Average,' or 'New.' This classification adds insights into the age distribution within each ZIP code, enhancing the overall housing analysis.

In [13]:
# Ensure YR_BUILT and YR_REMODEL are numeric and replace years > 2024 with NaN
df['YR_BUILT'] = pd.to_numeric(df['YR_BUILT'], errors='coerce')
df['YR_REMODEL'] = pd.to_numeric(df['YR_REMODEL'], errors='coerce')
df['YR_BUILT'] = df['YR_BUILT'].apply(lambda x: x if x <= 2024 else np.nan)
df['YR_REMODEL'] = df['YR_REMODEL'].apply(lambda x: x if x <= 2024 else np.nan)

# Group by ZIP_CODE and calculate the mean for YR_BUILT and YR_REMODEL
condition_analysis = df.groupby('ZIP_CODE').agg(
    yr_built_mean=('YR_BUILT', 'mean'),
    yr_remodel_mean=('YR_REMODEL', 'mean')
).reset_index()

# Define thresholds for building classification
def classify_building(yr_built, yr_remodel, old_threshold=1970, new_threshold=2000):
    """
    Classify building as 'Old', 'Average', or 'New' based on YR_REMODEL or YR_BUILT.
    If YR_REMODEL exists, use it; otherwise, use YR_BUILT.
    """
    if not pd.isna(yr_remodel):
        year = yr_remodel  # Use YR_REMODEL if available
    else:
        year = yr_built  # Otherwise, use YR_BUILT
    
    if pd.isna(year):
        return 'Unknown'
    elif year <= old_threshold:
        return 'Old'
    elif year >= new_threshold:
        return 'New'
    else:
        return 'Average'
    
# Apply classification based on YR_BUILT and YR_REMODEL
condition_analysis['building_classification'] = condition_analysis.apply(
    lambda row: classify_building(row['yr_built_mean'], row['yr_remodel_mean']), axis=1)

# Print the classification results based on the YR_BUILT and YR_REMODEL means
print("Building Classification Based on YR_BUILT and YR_REMODEL:")
print(condition_analysis[['ZIP_CODE', 'yr_built_mean', 'yr_remodel_mean', 'building_classification']])

Building Classification Based on YR_BUILT and YR_REMODEL:
    ZIP_CODE  yr_built_mean  yr_remodel_mean building_classification
0     2018.0    1971.000000      2016.000000                     New
1     2026.0    1949.571429      1995.500000                 Average
2     2108.0    1907.691133      1996.791639                 Average
3     2109.0    1925.371549      1995.711314                 Average
4     2110.0    1969.866667      1999.440191                 Average
5     2111.0    1964.257515      1998.628196                 Average
6     2112.0            NaN              NaN                 Unknown
7     2113.0    1914.214422      1995.828083                 Average
8     2114.0    1930.993818      1995.012387                 Average
9     2115.0    1918.874305      1993.881894                 Average
10    2116.0    1920.421059      1997.308711                 Average
11    2118.0    1929.833477      2000.275573                     New
12    2119.0    1927.842714      2000.844100 

Step 8: Include Address Details and Finalize Output
This final step merges street address details, adds overall condition labels, assigns a constant YEAR value, and saves the output to an Excel file for further analysis.

In [14]:
# Assuming 'ST_NUM' and 'ST_NAME' are part of the original dataset
# Extract those columns from the original dataset
st_num_name = df[['ZIP_CODE', 'ST_NUM', 'ST_NAME']].drop_duplicates()

# Merge 'st_num_name' with 'condition_analysis' to include 'ST_NUM' and 'ST_NAME' with the results
condition_analysis = pd.merge(condition_analysis, st_num_name, on='ZIP_CODE', how='left')

# Add overall condition label based on previous analysis
# Ensure that the column 'overall_cond_label' from earlier condition analysis is merged correctly
overall_cond_summary = df.groupby('ZIP_CODE').agg(
    overall_cond_mean=('OVERALL_COND_NUM', 'mean'),
).reset_index()

# Reapply the condition label mapping function to map numeric values to condition labels
def mean_to_condition(mean_score):
    if mean_score >= 4.75:
        return 'Excellent'
    elif mean_score >= 4:
        return 'Very Good'
    elif mean_score >= 3.5:
        return 'Good'
    elif mean_score >= 3:
        return 'Average'
    elif mean_score >= 2:
        return 'Fair'
    elif mean_score >= 1:
        return 'Poor'
    elif mean_score >= 0.5:
        return 'Very Poor'
    else:
        return 'Unsound'

overall_cond_summary['overall_cond_label'] = overall_cond_summary['overall_cond_mean'].apply(mean_to_condition)

# Merge overall condition labels with the condition_analysis DataFrame
condition_analysis = pd.merge(condition_analysis, overall_cond_summary[['ZIP_CODE', 'overall_cond_label']], on='ZIP_CODE', how='left')

# Assign a constant value for 'YEAR'
condition_analysis['YEAR'] = 2020

# Rearranging the columns as requested
final_output = condition_analysis[['YEAR', 'ST_NUM', 'ST_NAME', 'ZIP_CODE', 'overall_cond_label', 'building_classification']]

# Save the final result to a new Excel file
output_file_path = 'Property_Assessment_2020_Output.xlsx'
final_output.to_excel(output_file_path, index=False)

print(f"File saved successfully to {output_file_path}")

File saved successfully to Property_Assessment_2020_Output.xlsx
